# Imports & Setup

In [1]:
%load_ext autoreload
%autoreload 2

from datetime import date, datetime, timedelta
from functools import partial
from typing import Dict, Optional

import numpy as np
import polars as pl
from tqdm import tqdm
import matplotlib.pyplot as plt

from okx.store import OrderbookStore
from okx.recipes.forwards import build_forwards_pchip, build_forwards_kalman, assign_forwards
from okx.recipes.options import prepare_options
from evaluation.forwards_eval import evaluate_parity, summarize_parity, evaluate_pillar_fit, evaluate_loeo, summarize_pillar_fit, run_full_evaluation

from evaluation.plotting import plot_error_histogram, plot_error_over_time, plot_error_by_group

store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite"
)

# OB Data Fetch, Preprocessing, and Merging

### 1. Define parameters

In [2]:
def make_dates(start_date, n_days):
    dates = [start_date + timedelta(days=i) for i in range(n_days)]
    return dates
binning = '10s'
shared_params = {
    'inst_family': 'BTC-USD',
    'dates': make_dates(date(2025, 9, 1), 3),
    'verbose': True,
    'binning': binning,
    'cache_name': f'arb_check_{binning}',
    'batch_days': 5
}

In [27]:
store.clear_cache()

Cleared all caches


### 2. Fetch futures & options data, preprocess, merge, and calculate moneyness

In [4]:
futures_features = ['trim', 'strip', 'bin_ff', 'sink_bins', 'mid', 'tenor']
futures_lf = store.get(
    inst_type='FUTURES',
    depth=1,
    **shared_params,
    features=futures_features,
).rename({'bid_1_px': 'F_bid', 'ask_1_px': 'F_ask', 'mid': 'F_mid'}).drop('symbol')
print(f"Loaded {futures_lf.select(pl.len()).collect().item()} futures")
print(f"Futures columns: {futures_lf.collect_schema().names()}")
options_features = ['trim', 'strip', 'nullify', 'bin_ff', 'sink_bins', 'drop_nulls_strict', 'parse_option', 'tenor']
options_lf = store.get(
    inst_type='OPTION',
    depth=1,
    **shared_params,
    features=options_features,
)
print(f"Loaded {options_lf.select(pl.len()).collect().item()} options")
print(f"Options columns: {options_lf.collect_schema().names()}")

# Pair options
from okx.recipes.helpers import pair_options
options_lf = pair_options(options_lf)
print(f"Paired options: {options_lf.select(pl.len()).collect().item()}")
print(f"Paired options columns: {options_lf.collect_schema().names()}")

# Join and add moneyness
merged = futures_lf.join(options_lf, on=['timeMs', 'expiry', 'T'], how='inner')
merged = merged.with_columns((pl.col('strike') / pl.col('F_mid')).log().alias('moneyness'))
print(f"Merged: {merged.select(pl.len()).collect().item()}")
print(f"Merged columns: {merged.collect_schema().names()}")


[store] Getting BTC-USD/FUTURES for 3 dates (depth=1, 10s binning, 6 features)
Loaded 181433 futures
Futures columns: ['timeMs', 'F_bid', 'F_ask', 'F_mid', 'expiry', 'T']
[store] Getting BTC-USD/OPTION for 3 dates (depth=1, 10s binning, 8 features)
Loaded 14557214 options
Options columns: ['symbol', 'timeMs', 'bid_1_px', 'ask_1_px', 'strike', 'opt_type', 'expiry', 'T']
Paired options: 6112363
Paired options columns: ['timeMs', 'expiry', 'strike', 'T', 'call_bid', 'call_ask', 'put_bid', 'put_ask']
Merged: 4355373
Merged columns: ['timeMs', 'F_bid', 'F_ask', 'F_mid', 'expiry', 'T', 'strike', 'call_bid', 'call_ask', 'put_bid', 'put_ask', 'moneyness']


In [10]:
print(merged.filter(pl.col('moneyness') < -0.1).select('T', 'F_mid', 'strike', 'moneyness', 'call_ask', 'call_bid', 'BTC-USD').head(10).collect())

shape: (10, 7)
┌──────────┬───────────┬────────┬───────────┬──────────┬──────────┬───────────┐
│ T        ┆ F_mid     ┆ strike ┆ moneyness ┆ call_ask ┆ call_bid ┆ BTC-USD   │
│ ---      ┆ ---       ┆ ---    ┆ ---       ┆ ---      ┆ ---      ┆ ---       │
│ f64      ┆ f64       ┆ i64    ┆ f64       ┆ f64      ┆ f64      ┆ f64       │
╞══════════╪═══════════╪════════╪═══════════╪══════════╪══════════╪═══════════╡
│ 0.068429 ┆ 109922.85 ┆ 50000  ┆ -0.787756 ┆ 0.5465   ┆ 0.5445   ┆ 109507.75 │
│ 0.068429 ┆ 109896.15 ┆ 50000  ┆ -0.787513 ┆ 0.5465   ┆ 0.5445   ┆ 109479.95 │
│ 0.068428 ┆ 109921.85 ┆ 50000  ┆ -0.787747 ┆ 0.546    ┆ 0.5445   ┆ 109507.65 │
│ 0.068428 ┆ 109950.85 ┆ 50000  ┆ -0.78801  ┆ 0.546    ┆ 0.5445   ┆ 109533.95 │
│ 0.068428 ┆ 109964.75 ┆ 50000  ┆ -0.788137 ┆ 0.546    ┆ 0.5445   ┆ 109539.95 │
│ 0.068428 ┆ 109964.75 ┆ 50000  ┆ -0.788137 ┆ 0.546    ┆ 0.5445   ┆ 109539.95 │
│ 0.068427 ┆ 109964.75 ┆ 50000  ┆ -0.788137 ┆ 0.546    ┆ 0.5445   ┆ 109539.95 │
│ 0.068427 ┆ 109944.85 ┆ 

### 3. Add BTC-USD column from spot for numeraire conversion 

In [9]:
spot_lf = store.get(
    inst_type='SPOT',
    depth=0,
    **shared_params,
    features=['trim', 'strip', 'bin_ff', 'sink_bins']
).drop('symbol').rename({'mid': 'BTC-USD'})
print(f"Spot: {spot_lf.select(pl.len()).collect().item()}")
print(f"Spot columns: {spot_lf.collect_schema().names()}")

# Add BTC-USD column
merged = merged.join(spot_lf, on=['timeMs'], how='left')


[store] Getting BTC-USD/SPOT for 3 dates (depth=0, 10s binning, 4 features)
Spot: 25919
Spot columns: ['timeMs', 'BTC-USD']


In [5]:
print(merged.collect_schema().names())

['timeMs', 'F_bid', 'F_ask', 'F_mid', 'expiry', 'T', 'strike', 'call_bid_1_px', 'call_ask_1_px', 'put_bid_1_px', 'put_ask_1_px', 'moneyness', 'BTC-USD']


# Synthetic Futures
We will define prices for a number of order types for synthetic futures longs and shorts.

Using put-call parity: **F = K + C - P**

### Synthetic Futures Longs (Buy Call + Sell Put):
- **LONG_LMT_Cb_Pa**: limit order - buy call @ bid, sell put @ ask (best possible fill)
- **LONG_SYM_Cb_Pb**: symmetric - buy call @ bid, sell put @ bid
- **LONG_SYM_Ca_Pa**: symmetric - buy call @ ask, sell put @ ask
- **LONG_MKT_Ca_Pb**: market order - buy call @ ask, sell put @ bid (worst possible fill)

### Synthetic Futures Shorts (Sell Call + Buy Put):
- **SHORT_LMT_Ca_Pb**: limit order - sell call @ ask, buy put @ bid (best possible fill)
- **SHORT_SYM_Ca_Pa**: symmetric - sell call @ ask, buy put @ ask
- **SHORT_SYM_Cb_Pb**: symmetric - sell call @ bid, buy put @ bid
- **SHORT_MKT_Cb_Pa**: market order - sell call @ bid, buy put @ ask (worst possible fill)

### Arbitrage Logic:
- **Long Synthetic**: Profit if synthetic price < futures price → buy synthetic, sell futures
- **Short Synthetic**: Profit if synthetic price > futures price → sell synthetic, buy futures

In [6]:
# Define synthetic futures order types, refactored for clarity and two-column output (fwd, cost) per type

maker_fee = -0.0001  # -0.01% rebate
taker_fee = 0.00013  # 0.013% fee

def synth_cols(name, direction, call_order, put_order, description, execution):
    # Decide which price columns to use
    if direction == 'long':  # buy call, sell put
        call_px = 'call_bid_1_px' if call_order == 'maker' else 'call_ask_1_px'
        put_px  = 'put_ask_1_px' if put_order  == 'maker' else 'put_bid_1_px'
        call_fee_rate = maker_fee if call_order == 'maker' else taker_fee
        put_fee_rate  = maker_fee if put_order  == 'maker' else taker_fee
        # Option cost in USD (contract * conversion ratio)
        call_amt = pl.col(call_px) * pl.col('BTC-USD')
        put_amt  = pl.col(put_px)  * pl.col('BTC-USD')
        # Fee in USD
        call_fee = call_amt * call_fee_rate
        put_fee  = put_amt  * put_fee_rate
        # Total synthetic forward
        fwd_expr = pl.col("strike") + (call_amt - put_amt)
        fee_expr = (call_fee + put_fee)
    else:  # short: sell call, buy put
        call_px = 'call_ask_1_px' if call_order == 'maker' else 'call_bid_1_px'
        put_px  = 'put_bid_1_px' if put_order  == 'maker' else 'put_ask_1_px'
        call_fee_rate = maker_fee if call_order == 'maker' else taker_fee
        put_fee_rate  = maker_fee if put_order  == 'maker' else taker_fee
        call_amt = pl.col(call_px) * pl.col('BTC-USD')
        put_amt  = pl.col(put_px)  * pl.col('BTC-USD')
        call_fee = call_amt * call_fee_rate
        put_fee  = put_amt  * put_fee_rate
        fwd_expr = pl.col("strike") - (put_amt - call_amt)
        fee_expr = (call_fee + put_fee)
    return [
        fwd_expr.alias(f"{name}_fwd"),
        fee_expr.alias(f"{name}_cost")
    ]

# Order definitions: each is (direction, call_order, put_order, description, execution)
SYNTHETIC_ORDER_DEFS = [
    # LONG
    ('LONG_LMT_Cb_Pa',   'long',  'maker', 'maker',  'Long limit: buy call @ bid, sell put @ ask',     'limit'),
    ('LONG_SYM_Cb_Pb',   'long',  'maker', 'taker',  'Long sym: buy call @ bid, sell put @ bid',       'symmetric'),
    ('LONG_SYM_Ca_Pa',   'long',  'taker', 'maker',  'Long sym: buy call @ ask, sell put @ ask',       'symmetric'),
    ('LONG_MKT_Ca_Pb',   'long',  'taker', 'taker',  'Long market: buy call @ ask, sell put @ bid',    'market'),
    # SHORT
    ('SHORT_LMT_Ca_Pb',  'short', 'maker', 'maker',  'Short limit: sell call @ ask, buy put @ bid',    'limit'),
    ('SHORT_SYM_Ca_Pa',  'short', 'maker', 'taker',  'Short sym: sell call @ ask, buy put @ ask',      'symmetric'),
    ('SHORT_SYM_Cb_Pb',  'short', 'taker', 'maker',  'Short sym: sell call @ bid, buy put @ bid',      'symmetric'),
    ('SHORT_MKT_Cb_Pa',  'short', 'taker', 'taker',  'Short market: sell call @ bid, buy put @ ask',   'market'),
]

# Build mapping for order type metadata (for lookup/use if desired)
SYNTHETIC_ORDER_TYPES = {
    name: dict(
        direction=direction,
        call_order=call_order,
        put_order=put_order,
        description=description,
        execution=execution,
    )
    for name, direction, call_order, put_order, description, execution in SYNTHETIC_ORDER_DEFS
}

# Build all synthetic columns (forward and fee for each type)
synthetic_columns = []
for name, direction, call_order, put_order, description, execution in SYNTHETIC_ORDER_DEFS:
    synthetic_columns.extend(synth_cols(name, direction, call_order, put_order, description, execution))

merged = merged.with_columns(synthetic_columns)



In [16]:
# Add profit, cost, return, and annualized return columns for each order type

arbitrage_columns = []

for name, direction, call_order, put_order, description, execution in SYNTHETIC_ORDER_DEFS:
    fwd_col = f"{name}_fwd"
    cost_col = f"{name}_cost"
    
    if direction == 'long':
        # LONG synthetic + SHORT futures arbitrage
        # Profit = F_bid - synthetic_fwd - option_costs - futures_fee
        futures_fee = pl.col('F_bid') * taker_fee
        profit = pl.col('F_bid') - pl.col(fwd_col) - pl.col(cost_col) - futures_fee
        # Capital at risk = net option premium paid (absolute value)
        capital = pl.col(fwd_col) - pl.col('strike') + pl.col(cost_col) + futures_fee
    else:
        # SHORT synthetic + LONG futures arbitrage
        # Profit = synthetic_fwd - F_ask - option_costs - futures_fee
        futures_fee = pl.col('F_ask') * taker_fee
        profit = pl.col(fwd_col) - pl.col('F_ask') - pl.col(cost_col) - futures_fee
        # Capital at risk = net option premium received (absolute value)
        capital = (pl.col(fwd_col) - pl.col('strike')) * -1 + pl.col(cost_col) + futures_fee
    
    # Raw return (can be negative or infinite if capital near zero)
    # When capital is negative (net credit received), return is not meaningful
    raw_return = pl.when(capital > 0).then(profit / capital).otherwise(pl.lit(float('nan')))

    # Clip raw return to reasonable bounds (e.g., -10 to 10 = -1000% to 1000%)
    raw_return_clipped = raw_return.clip(-10, 10)

    # Annualized return: (1 + return)^(1/T) - 1
    # Only calculate when raw_return is valid
    annualized_return = pl.when(raw_return_clipped.is_not_nan()).then((1 + raw_return_clipped).pow(1 / pl.col('T')) - 1).otherwise(pl.lit(float('nan')))
    
    # Profitable flag
    profitable = profit > 0
    
    arbitrage_columns.extend([
        profit.alias(f'{name}_profit'),
        capital.alias(f'{name}_capital'),
        raw_return.alias(f'{name}_return'),
        annualized_return.alias(f'{name}_annual_return'),
        profitable.alias(f'{name}_profitable')
    ])

merged = merged.with_columns(arbitrage_columns)

print(f"Added arbitrage metrics for {len(SYNTHETIC_ORDER_DEFS)} order types")
print(f"Total columns: {len(merged.collect_schema().names())}")
print(f"\nNew columns per order type: _profit, _capital, _return, _annual_return, _profitable")


Added arbitrage metrics for 8 order types
Total columns: 69

New columns per order type: _profit, _capital, _return, _annual_return, _profitable


In [17]:
# Summarize arbitrage opportunities across all order types

def summarize_arbitrage(lf, order_types_dict):
    """
    Summarize arbitrage statistics for all order types.
    Returns a dictionary with statistics for each order type.
    """
    results = {}
    
    for order_type_name, order_type_info in order_types_dict.items():
        profitable_col = f'{order_type_name}_profitable'
        profit_col = f'{order_type_name}_profit'
        return_col = f'{order_type_name}_return'
        annual_return_col = f'{order_type_name}_annual_return'
        
        stats = lf.select([
            # Count and frequency of profitable opportunities
            pl.col(profitable_col).sum().alias('count'),
            pl.col(profitable_col).mean().alias('frequency'),
            # Profit statistics (when profitable)
            pl.when(pl.col(profitable_col))
              .then(pl.col(profit_col))
              .mean()
              .alias('avg_profit'),
            pl.when(pl.col(profitable_col))
              .then(pl.col(profit_col))
              .max()
              .alias('max_profit'),
            # Return statistics (when profitable and capital > 0)
            pl.when(pl.col(profitable_col) & pl.col(return_col).is_not_nan())
              .then(pl.col(return_col))
              .mean()
              .alias('avg_return'),
            pl.when(pl.col(profitable_col) & pl.col(annual_return_col).is_not_nan())
              .then(pl.col(annual_return_col))
              .mean()
              .alias('avg_annual_return'),
        ]).collect()
        
        results[order_type_name] = {
            'direction': order_type_info['direction'],
            'execution': order_type_info['execution'],
            'description': order_type_info['description'],
            'count': stats['count'][0],
            'frequency': stats['frequency'][0],
            'avg_profit': stats['avg_profit'][0],
            'max_profit': stats['max_profit'][0],
            'avg_return': stats['avg_return'][0],
            'avg_annual_return': stats['avg_annual_return'][0],
        }
    
    return results

# Get summary
arb_summary = summarize_arbitrage(merged, SYNTHETIC_ORDER_TYPES)

# Display results
print("=== ARBITRAGE OPPORTUNITY SUMMARY ===\n")
for order_type_name, stats in arb_summary.items():
    print(f"{order_type_name}:")
    print(f"  {stats['description']}")
    print(f"  Count: {stats['count']:,} | Frequency: {stats['frequency']:.2%}")
    print(f"  Avg Profit: ${stats['avg_profit']:.2f} | Max Profit: ${stats['max_profit']:.2f}")
    print(f"  Avg Return: {stats['avg_return']:.2%} | Avg Annual Return: {stats['avg_annual_return']:.2%}")
    print()


=== ARBITRAGE OPPORTUNITY SUMMARY ===

LONG_LMT_Cb_Pa:
  Long limit: buy call @ bid, sell put @ ask
  Count: 3,503,867 | Frequency: 80.45%
  Avg Profit: $1294.78 | Max Profit: $111869.93
  Avg Return: 701.69% | Avg Annual Return: 984175746983391156971091264999353922609540036666658394248568042183739296022198485423102970133715044482040865557369076402223631413253892029400016755204821661929129073062025702384963603467971931334744295450212395182820031978717165902799010585649926934470973338713791076746611235225600.00%

LONG_SYM_Cb_Pb:
  Long sym: buy call @ bid, sell put @ bid
  Count: 1,900,731 | Frequency: 43.64%
  Avg Profit: $2017.59 | Max Profit: $84858.18
  Avg Return: 553.35% | Avg Annual Return: 1067833464924823100638475445664648153062497777077526750746431933328227570870059815973971871253785424234829537577128748185591671225086308695839542881386496111972994446303365003822612965563880422863071564712637369491355247430267311187387080094054493421654718391375267976840568092229632.00%

LONG_

In [18]:
# Show top profitable opportunities for market orders (most conservative)
import polars as pl

# Set Polars to not abbreviate or truncate column output in print statements
pl.Config.set_tbl_cols(-1)  # Show all columns, no abbreviation

print("=== TOP LONG SYNTHETIC (Market Order) ARBITRAGE OPPORTUNITIES ===\n")
long_arbs = (
    merged
    .filter(pl.col('LONG_MKT_Ca_Pb_profitable'))
    .select([
        'timeMs', 'expiry', 'T', 'strike', 'moneyness',
        'F_bid', 'LONG_MKT_Ca_Pb_fwd', 'LONG_MKT_Ca_Pb_cost',
        'LONG_MKT_Ca_Pb_profit', 'LONG_MKT_Ca_Pb_return', 'LONG_MKT_Ca_Pb_capital', 'LONG_MKT_Ca_Pb_annual_return'
    ])
    .sort('LONG_MKT_Ca_Pb_profit', descending=True)
    .head(10)
    .collect()
)
print(long_arbs)

print("\n=== TOP SHORT SYNTHETIC (Market Order) ARBITRAGE OPPORTUNITIES ===\n")
short_arbs = (
    merged
    .filter(pl.col('SHORT_MKT_Cb_Pa_profitable'))
    .select([
        'timeMs', 'expiry', 'T', 'strike', 'moneyness',
        'F_ask', 'SHORT_MKT_Cb_Pa_fwd', 'SHORT_MKT_Cb_Pa_cost',
        'SHORT_MKT_Cb_Pa_profit', 'SHORT_MKT_Cb_Pa_return', 'SHORT_MKT_Cb_Pa_capital', 'SHORT_MKT_Cb_Pa_annual_return'
    ])
    .sort('SHORT_MKT_Cb_Pa_profit', descending=True)
    .head(10)
    .collect()
)
print(short_arbs)


=== TOP LONG SYNTHETIC (Market Order) ARBITRAGE OPPORTUNITIES ===

shape: (10, 12)
┌────────┬────────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ timeMs ┆ expiry ┆ T      ┆ strik ┆ money ┆ F_bid ┆ LONG_ ┆ LONG_ ┆ LONG_ ┆ LONG_ ┆ LONG_ ┆ LONG_ │
│ ---    ┆ ---    ┆ ---    ┆ e     ┆ ness  ┆ ---   ┆ MKT_C ┆ MKT_C ┆ MKT_C ┆ MKT_C ┆ MKT_C ┆ MKT_C │
│ i64    ┆ i64    ┆ f64    ┆ ---   ┆ ---   ┆ f64   ┆ a_Pb_ ┆ a_Pb_ ┆ a_Pb_ ┆ a_Pb_ ┆ a_Pb_ ┆ a_Pb_ │
│        ┆        ┆        ┆ i64   ┆ f64   ┆       ┆ fwd   ┆ cost  ┆ profi ┆ retur ┆ capit ┆ annua │
│        ┆        ┆        ┆       ┆       ┆       ┆ ---   ┆ ---   ┆ t     ┆ n     ┆ al    ┆ l_ret │
│        ┆        ┆        ┆       ┆       ┆       ┆ f64   ┆ f64   ┆ ---   ┆ ---   ┆ ---   ┆ urn   │
│        ┆        ┆        ┆       ┆       ┆       ┆       ┆       ┆ f64   ┆ f64   ┆ f64   ┆ ---   │
│        ┆        ┆        ┆       ┆       ┆       ┆       ┆       ┆       ┆       ┆       ┆ f64   │
╞═══════

In [13]:
# Show top profitable opportunities for market orders (most conservative)
import polars as pl

# Set Polars to not abbreviate or truncate column output in print statements
pl.Config.set_tbl_cols(-1)  # Show all columns, no abbreviation

print("=== TOP LONG SYNTHETIC (Market Order) ARBITRAGE OPPORTUNITIES ===\n")
long_arbs = (
    merged
    .filter(pl.col('LONG_MKT_Ca_Pb_profitable'))
    .select([
        'timeMs', 'expiry', 'T', 'strike', 'moneyness',
        'F_bid', 'LONG_MKT_Ca_Pb_fwd', 'LONG_MKT_Ca_Pb_cost',
        'LONG_MKT_Ca_Pb_profit', 'LONG_MKT_Ca_Pb_return', 'LONG_MKT_Ca_Pb_capital', 'LONG_MKT_Ca_Pb_annual_return'
    ])
    .sort('LONG_MKT_Ca_Pb_capital')
    .head(10)
    .collect()
)
print(long_arbs)

print("\n=== TOP SHORT SYNTHETIC (Market Order) ARBITRAGE OPPORTUNITIES ===\n")
short_arbs = (
    merged
    .filter(pl.col('SHORT_MKT_Cb_Pa_profitable'))
    .select([
        'timeMs', 'expiry', 'T', 'strike', 'moneyness',
        'F_ask', 'SHORT_MKT_Cb_Pa_fwd', 'SHORT_MKT_Cb_Pa_cost',
        'SHORT_MKT_Cb_Pa_profit', 'SHORT_MKT_Cb_Pa_return', 'SHORT_MKT_Cb_Pa_capital', 'SHORT_MKT_Cb_Pa_annual_return'
    ])
    .sort('SHORT_MKT_Cb_Pa_capital')
    .head(10)
    .collect()
)
print(short_arbs)


=== TOP LONG SYNTHETIC (Market Order) ARBITRAGE OPPORTUNITIES ===

shape: (10, 12)
┌────────┬────────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ timeMs ┆ expiry ┆ T      ┆ strik ┆ money ┆ F_bid ┆ LONG_ ┆ LONG_ ┆ LONG_ ┆ LONG_ ┆ LONG_ ┆ LONG_ │
│ ---    ┆ ---    ┆ ---    ┆ e     ┆ ness  ┆ ---   ┆ MKT_C ┆ MKT_C ┆ MKT_C ┆ MKT_C ┆ MKT_C ┆ MKT_C │
│ i64    ┆ i64    ┆ f64    ┆ ---   ┆ ---   ┆ f64   ┆ a_Pb_ ┆ a_Pb_ ┆ a_Pb_ ┆ a_Pb_ ┆ a_Pb_ ┆ a_Pb_ │
│        ┆        ┆        ┆ i64   ┆ f64   ┆       ┆ fwd   ┆ cost  ┆ profi ┆ retur ┆ capit ┆ annua │
│        ┆        ┆        ┆       ┆       ┆       ┆ ---   ┆ ---   ┆ t     ┆ n     ┆ al    ┆ l_ret │
│        ┆        ┆        ┆       ┆       ┆       ┆ f64   ┆ f64   ┆ ---   ┆ ---   ┆ ---   ┆ urn   │
│        ┆        ┆        ┆       ┆       ┆       ┆       ┆       ┆ f64   ┆ f64   ┆ f64   ┆ ---   │
│        ┆        ┆        ┆       ┆       ┆       ┆       ┆       ┆       ┆       ┆       ┆ f64   │
╞═══════

In [12]:
# Summarize arbitrage opportunities across all order types
def summarize_arbitrage_opportunities(df, order_types_dict):
    """
    Summarize arbitrage statistics for all order types (using annualized return).
    
    Returns a dictionary with statistics for each order type.
    """
    results = {}
    
    for order_type_name, order_type_info in order_types_dict.items():
        return_col = f'{order_type_name}_return'
        arb_col = f'{order_type_name}_arbitrage'
        
        stats = df.select([
            pl.col(arb_col).sum().alias('count'),
            pl.col(arb_col).mean().alias('frequency'),
            pl.when(pl.col(arb_col))
              .then(pl.col(return_col))
              .mean()
              .alias('avg_return'),
            pl.when(pl.col(arb_col))
              .then(pl.col(return_col))
              .max()
              .alias('max_return'),
            pl.col(return_col).mean().alias('avg_return_all'),
        ]).collect()
        
        results[order_type_name] = {
            'direction': order_type_info['direction'],
            'execution': order_type_info['execution'],
            'description': order_type_info['description'],
            'count': stats['count'][0],
            'frequency': stats['frequency'][0],
            'avg_return': stats['avg_return'][0],
            'max_return': stats['max_return'][0],
            'avg_return_all': stats['avg_return_all'][0],
        }
    
    return results

# Get summary
arb_summary = summarize_arbitrage_opportunities(merged, SYNTHETIC_ORDER_TYPES)

# Display results
print("\n=== ARBITRAGE OPPORTUNITY SUMMARY (Annualized Return) ===\n")
for order_type_name, stats in arb_summary.items():
    print(f"{order_type_name}:")
    print(f"  {stats['description']}")
    print(f"  Count: {stats['count']:,} | Frequency: {stats['frequency']:.4%}")
    print(f"  Avg Annualized Return (when arb): {stats['avg_return']:.4%}")
    print(f"  Max Annualized Return: {stats['max_return']:.4%}")
    print(f"  Avg Annualized Return (all): {stats['avg_return_all']:.4%}")
    print()


ColumnNotFoundError: unable to find column "LONG_LMT_Cb_Pa_arbitrage"; valid columns: ["timeMs", "F_bid", "F_ask", "F_mid", "expiry", "T", "strike", "call_bid_1_px", "call_ask_1_px", "put_bid_1_px", "put_ask_1_px", "moneyness", "BTC-USD", "LONG_LMT_Cb_Pa_fwd", "LONG_LMT_Cb_Pa_cost", "LONG_SYM_Cb_Pb_fwd", "LONG_SYM_Cb_Pb_cost", "LONG_SYM_Ca_Pa_fwd", "LONG_SYM_Ca_Pa_cost", "LONG_MKT_Ca_Pb_fwd", "LONG_MKT_Ca_Pb_cost", "SHORT_LMT_Ca_Pb_fwd", "SHORT_LMT_Ca_Pb_cost", "SHORT_SYM_Ca_Pa_fwd", "SHORT_SYM_Ca_Pa_cost", "SHORT_SYM_Cb_Pb_fwd", "SHORT_SYM_Cb_Pb_cost", "SHORT_MKT_Cb_Pa_fwd", "SHORT_MKT_Cb_Pa_cost", "LONG_LMT_Cb_Pa_profit", "LONG_LMT_Cb_Pa_capital", "LONG_LMT_Cb_Pa_return", "LONG_LMT_Cb_Pa_annual_return", "LONG_LMT_Cb_Pa_profitable", "LONG_SYM_Cb_Pb_profit", "LONG_SYM_Cb_Pb_capital", "LONG_SYM_Cb_Pb_return", "LONG_SYM_Cb_Pb_annual_return", "LONG_SYM_Cb_Pb_profitable", "LONG_SYM_Ca_Pa_profit", "LONG_SYM_Ca_Pa_capital", "LONG_SYM_Ca_Pa_return", "LONG_SYM_Ca_Pa_annual_return", "LONG_SYM_C